In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
data = pd.read_csv("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/sample_submission.csv")
data.to_csv("submission.csv",index=False)

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
import librosa.display
import matplotlib.pyplot as plt
import random
import torch

import warnings
warnings.filterwarnings("ignore")

In [ ]:
#----------------------------- DON'T CHANGE THIS --------------------------
DATA_SEED = 67
TRAINING_SEED = 1234
SR = 22050
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB=20
TARGET_SNR_DB = 10

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)
torch.cuda.manual_seed(DATA_SEED)

In [ ]:
# CONFIGURATION
DATA_ROOT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
GENRES = [
    'blues', 'classical', 'country', 'disco', 'hiphop', 
    'jazz', 'metal', 'pop', 'reggae', 'rock'
]
STEMS = {
    'drums.wav': 'drums',
    'vocals.wav': 'vocals',
    'bass.wav': 'bass',
    'other.wav': 'other'
}
STEM_KEYS = ['drums', 'vocals', 'bass', 'other']
GENRE_TO_TEST = 'rock'
SONG_INDEX = 0

In [ ]:
def build_dataset(root_dir, val_split=0.17, seed=42):
    train_dataset = {g: {s.replace('.wav', ''): [] for s in STEM_KEYS} for g in GENRES}
    val_dataset   = {g: {s.replace('.wav', ''): [] for s in STEM_KEYS} for g in GENRES}

    rng = random.Random(seed)
    
    # Tracking variables for Q1 & Q2
    corrupted_count = 0
    less_than_5_0491MB = 0
    greater_than_5_0493MB = 0
    
    # 1MB = 1024 * 1024 bytes
    threshold_low = 5.0491 * 1024 * 1024
    threshold_high = 5.0493 * 1024 * 1024

    for genre in GENRES:
        genre_path = os.path.join(root_dir, 'genres_stems', genre)
        if not os.path.exists(genre_path): continue
        
        # Get list of song folders
        songs = sorted([d for d in os.listdir(genre_path) if os.path.isdir(os.path.join(genre_path, d))])
        valid_songs = []

        for song in songs:
            song_path = os.path.join(genre_path, song)
            stems_present = True
            
            for s_key in STEM_KEYS:
                f_path = os.path.join(song_path, f"{s_key}.wav")
                if not os.path.exists(f_path):
                    stems_present = False
                    break
                
                f_size = os.path.getsize(f_path)
                if f_size < 4096: # 4kb
                    corrupted_count += 1
                    stems_present = False
                
                if f_size < threshold_low: less_than_5_0491MB += 1
                if f_size > threshold_high: greater_than_5_0493MB += 1
            
            if stems_present:
                valid_songs.append(song)

        # Shuffle and Split
        rng.shuffle(valid_songs)
        split_point = int(len(valid_songs) * (1 - val_split))
        tr_songs, val_songs = valid_songs[:split_point], valid_songs[split_point:]

        for s in tr_songs:
            for k in STEM_KEYS:
                train_dataset[genre][k].append(os.path.join(genre_path, s, f"{k}.wav"))
        for s in val_songs:
            for k in STEM_KEYS:
                val_dataset[genre][k].append(os.path.join(genre_path, s, f"{k}.wav"))

    print(f"Q1 Answer: {corrupted_count + less_than_5_0491MB}")
    print(f"Q2 Answer: {abs(greater_than_5_0493MB - less_than_5_0491MB)}")
    return train_dataset, val_dataset

tr, val = build_dataset(DATA_ROOT)
print(f"Q3 Answer: {abs(len(tr['reggae']['drums']) - len(val['country']['vocals']))}")

In [ ]:
def find_long_silences(dataset_dict, sr=SR, threshold_sec=DURATION, top_db=TOP_DB):
    records = []
    for genre, stems in tqdm(dataset_dict.items()):
        for stem_name, file_paths in stems.items():
            for path in file_paths:
                y, _ = librosa.load(path, sr=sr)
                duration = librosa.get_duration(y=y, sr=sr)
                intervals = librosa.effects.split(y, top_db=top_db)
                
                max_silence = 0
                locs = []
                
                if len(intervals) == 0:
                    max_silence = duration
                    locs = ["full"]
                else:
                    # Check Start/End/Middle
                    start_gap = intervals[0][0] / sr
                    end_gap = (len(y) - intervals[-1][1]) / sr
                    
                    if start_gap >= threshold_sec: locs.append("start")
                    if end_gap >= threshold_sec: locs.append("end")
                    
                    max_silence = max(start_gap, end_gap)
                    
                    for i in range(len(intervals)-1):
                        mid_gap = (intervals[i+1][0] - intervals[i][1]) / sr
                        if mid_gap >= threshold_sec:
                            if "middle" not in locs: locs.append("middle")
                        max_silence = max(max_silence, mid_gap)

                if max_silence >= threshold_sec:
                    records.append({
                        "Genre": genre, "Stem": stem_name, "Max_Silence_Sec": max_silence,
                        "Silence_Location": " ".join(locs)
                    })
    return pd.DataFrame(records)

In [ ]:
df_silence = find_long_silences(tr, threshold_sec=5.0, top_db=TOP_DB)

q4_ans = len(df_silence)

vocals_df = df_silence[df_silence['Stem'] == 'vocals']
q5_ans = len(vocals_df)

q6_ans = vocals_df['Max_Silence_Sec'].mean() if not vocals_df.empty else 0

jazz_drums = df_silence[(df_silence['Genre'] == 'jazz') & (df_silence['Stem'] == 'drums')]
q7_ans = len(jazz_drums)

jazz_drums_mid_only = jazz_drums[jazz_drums['Silence_Location'].str.strip() == 'middle']
q8_ans = len(jazz_drums_mid_only)

jazz_drums_10s = jazz_drums[jazz_drums['Max_Silence_Sec'] >= 10]
q9_ans = len(jazz_drums_10s)

print(f"Answer Q4: {q4_ans}")
print(f"Answer Q5: {q5_ans}")
print(f"Answer Q6: {round(q6_ans, 2)}")
print(f"Answer Q7: {q7_ans}")
print(f"Answer Q8: {q8_ans}")
print(f"Answer Q9: {q9_ans}")

In [ ]:
# Select first rock song from 'tr'
rock_song_paths = [tr['rock'][k][SONG_INDEX] for k in STEM_KEYS]
stems_audio = [librosa.load(p, sr=SR, duration=DURATION)[0] for p in rock_song_paths]

stems_stack = np.array(stems_audio)
mix_raw = np.sum(stems_stack, axis=0)

rms_val = np.sqrt(np.mean(mix_raw**2))
max_val = np.max(np.abs(mix_raw))
mix_norm = mix_raw / max_val

print(f"Q10: {len(mix_raw)}")
print(f"Q11: {round(rms_val, 2)}")
print(f"Q12: {round(max_val, 2)}")

In [ ]:
import librosa
import os

jazz_path = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/jazz/'
durations = []

for song_folder in os.listdir(jazz_path):
    # Path to one of the stems
    file_path = os.path.join(jazz_path, song_folder, 'drums.wav')
    if os.path.exists(file_path):
        durations.append(librosa.get_duration(path=file_path))

mean_duration = sum(durations) / len(durations)
print(f"Mean Jazz Duration: {mean_duration} seconds")

In [ ]:
import os
import soundfile as sf
from tqdm import tqdm

DATA_ROOT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
folders_to_scan = ['genres_stems', 'ESC-50-master/audio', 'mashups']

def get_unique_sample_rates(root_path, target_folders):
    unique_rates = set()
    
    for folder in target_folders:
        full_path = os.path.join(root_path, folder)
        print(f"Scanning {folder}...")
        
        # Walk through all subdirectories and files
        for root, dirs, files in os.walk(full_path):
            for file in files:
                if file.endswith('.wav'):
                    file_path = os.path.join(root, file)
                    try:
                        # sf.info is much faster than librosa.load
                        info = sf.info(file_path)
                        unique_rates.add(info.samplerate)
                    except Exception as e:
                        print(f"Error reading {file}: {e}")
                        
    return sorted(list(unique_rates))

# Execution
rates = get_unique_sample_rates(DATA_ROOT, folders_to_scan)
print(f"\nUnique Sample Rates: {rates}")

In [ ]:
import os
from tqdm import tqdm

DATA_ROOT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"

def count_empty_files(directory):
    empty_count = 0
    total_audio_files = 0
    
    # Walk through all genre folders and song subfolders
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith('.wav'):
                total_audio_files += 1
                file_path = os.path.join(root, file)
                
                # Check if file size is 0
                if os.path.getsize(file_path) == 0:
                    empty_count += 1
                    print(f"Empty file found: {file_path}")
                    
    return empty_count, total_audio_files

# Execution
empty_files, total_files = count_empty_files(DATA_ROOT)

print("-" * 30)
print(f"Total .wav files scanned: {total_files}")
print(f"Total empty (0-byte) files: {empty_files}")

In [ ]:
import os
import librosa
import numpy as np
from tqdm import tqdm

DATA_ROOT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"
genres = ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']

peak_values_db = []

for genre in genres:
    genre_path = os.path.join(DATA_ROOT, genre)
    song_folders = [d for d in os.listdir(genre_path) if os.path.isdir(os.path.join(genre_path, d))]
    
    for folder in song_folders:
        file_path = os.path.join(genre_path, folder, 'vocals.wav')
        
        if os.path.exists(file_path):
            # sr=None is faster as it skips resampling
            y, _ = librosa.load(file_path, sr=None)
            
            if len(y) > 0:
                # Find the peak absolute amplitude
                peak_amp = np.max(np.abs(y))
                
                if peak_amp > 1e-10: # Ignore absolute silence
                    # Convert to dB: 20 * log10(amplitude)
                    peak_db = 20 * np.log10(peak_amp)
                    peak_values_db.append(peak_db)

avg_peak_db = np.mean(peak_values_db)
print(f"Average Peak Amplitude: {avg_peak_db:.2f} dB")

In [ ]:
import os
import librosa
import numpy as np
import pandas as pd
from tqdm import tqdm

DATA_ROOT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"
GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']
STEM_FILES = ['bass.wav', 'drums.wav', 'other.wav', 'vocals.wav']

def get_genre_centroid_stats(data_root, genres):
    genre_means = {}
    
    for genre in genres:
        print(f"Processing {genre}...")
        genre_path = os.path.join(data_root, genre)
        song_folders = [d for d in os.listdir(genre_path) if os.path.isdir(os.path.join(genre_path, d))]
        
        song_centroids = []
        
        # Processing a sample (or all) to find the mean
        for folder in tqdm(song_folders[:20]): # Using 20 songs per genre for a quick statistical estimate
            full_signal = None
            
            for stem in STEM_FILES:
                path = os.path.join(genre_path, folder, stem)
                if os.path.exists(path):
                    y, sr = librosa.load(path, sr=22050)
                    if full_signal is None:
                        full_signal = y
                    else:
                        # Ensure signals match in length before summing
                        min_len = min(len(full_signal), len(y))
                        full_signal = full_signal[:min_len] + y[:min_len]
            
            if full_signal is not None and np.max(np.abs(full_signal)) > 1e-4:
                # Calculate Spectral Centroid
                sc = librosa.feature.spectral_centroid(y=full_signal, sr=sr)[0]
                # Filter out silence/NaNs
                rms = librosa.feature.rms(y=full_signal)[0]
                valid_sc = sc[rms > 0.001]
                
                if len(valid_sc) > 0:
                    song_centroids.append(np.mean(valid_sc))
        
        genre_means[genre] = np.mean(song_centroids)
    
    return genre_means

# Execute and find the maximum
results = get_genre_centroid_stats(DATA_ROOT, GENRES)
brightest_genre = max(results, key=results.get)

print("\n--- Results ---")
for g, val in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"{g.capitalize()}: {val:.2f} Hz")

print(f"\nWinner: {brightest_genre.upper()} has the highest mean spectral centroid.")

In [ ]:
import os
import librosa
import numpy as np
from tqdm import tqdm

DATA_ROOT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"
THRESHOLD = 1e-4
DURATION = 0.5  # seconds

silent_count = 0
total_files = 0

# Get all genre folders
genres = [d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d))]

for genre in genres:
    genre_path = os.path.join(DATA_ROOT, genre)
    # Walk through song folders and stems
    for root, dirs, files in os.walk(genre_path):
        for file in files:
            if file.endswith('.wav'):
                total_files += 1
                file_path = os.path.join(root, file)
                
                try:
                    # Load only the first 0.5 seconds
                    y, sr = librosa.load(file_path, sr=None, duration=DURATION)
                    
                    if len(y) > 0:
                        # Check if peak amplitude is below threshold
                        if np.max(np.abs(y)) < THRESHOLD:
                            silent_count += 1
                except Exception as e:
                    continue

print(f"\nTotal Files Scanned: {total_files}")
print(f"Files with silent intro (< {DURATION}s): {silent_count}")
print(f"Percentage: {(silent_count/total_files)*100:.2f}%")

In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, confusion_matrix, classification_report

# --- 1. Setup and Preprocessing ---
ROOT = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
STEMS_PATH = os.path.join(ROOT, 'genres_stems')
GENRES = ["blues", "classical", "country", "disco", "hiphop", "jazz", "metal", "pop", "reggae", "rock"]

def extract_features(song_path):
    # Load 10s at 22050Hz
    y, sr = librosa.load(os.path.join(song_path, 'other.wav'), sr=22050, duration=10)
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    spec_cent = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    zcr = np.mean(librosa.feature.zero_crossing_rate(y))
    rolloff = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))
    
    # tempo is an array in newer librosa versions, so we extract the scalar
    tempo_val = float(tempo[0]) if isinstance(tempo, np.ndarray) else float(tempo)
    return [tempo_val, spec_cent, zcr, rolloff]

# --- 2. Data Preparation & Stratified Split ---
data = []
for g in GENRES:
    gp = os.path.join(STEMS_PATH, g)
    # Note: os.listdir order is random. Sort it if you want reproducible subsets!
    songs = sorted([s for s in os.listdir(gp) if os.path.isdir(os.path.join(gp, s))])
    for s in songs[:50]: # Sampling 50 for speed; use all for final
        data.append({'path': os.path.join(gp, s), 'genre': g})

df = pd.DataFrame(data)
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['genre'], random_state=42)

# --- 3. Model Training (Decision Tree) ---
X_train = np.array([extract_features(p) for p in train_df['path']])
y_train = train_df['genre']
X_val = np.array([extract_features(p) for p in val_df['path']])
y_val = val_df['genre']

clf = DecisionTreeClassifier(max_depth=5, random_state=42)
clf.fit(X_train, y_train)

# --- 4. Validation & Metrics ---
y_pred = clf.predict(X_val)
macro_f1 = f1_score(y_val, y_pred, average='macro')
cm = confusion_matrix(y_val, y_pred, labels=GENRES)
cr = classification_report(y_val, y_pred, target_names=GENRES)

print(f"Validation Macro F1 Score: {macro_f1:.4f}\n")
print("Detailed Classification Report:")
print(cr)

# --- 5. Visualizing CM and Computing TP, TN, FP, FN ---


plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=GENRES, yticklabels=GENRES, cmap='Blues')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Genre Classification Confusion Matrix')
plt.show()

# Compute TP, TN, FP, FN across all classes
tp = np.diag(cm)
fp = cm.sum(axis=0) - tp
fn = cm.sum(axis=1) - tp
tn = cm.sum() - (fp + fn + tp)

print("\n--- Answers to your questions ---")
cr_dict = classification_report(y_val, y_pred, target_names=GENRES, output_dict=True)

print(f"1. Validation macro F1 score: {macro_f1:.4f}")
print(f"2. Precision of hiphop: {cr_dict['hiphop']['precision']:.4f}")
print(f"3. Recall of pop: {cr_dict['pop']['recall']:.4f}")
print(f"4. Model accuracy: {cr_dict['accuracy']:.4f}")

highest_tp_idx = np.argmax(tp)
lowest_fn_idx = np.argmin(fn)
print(f"5. Genre with the highest true positives: {GENRES[highest_tp_idx].capitalize()} (TP={tp[highest_tp_idx]})")
print(f"6. Genre with the lowest false negatives: {GENRES[lowest_fn_idx].capitalize()} (FN={fn[lowest_fn_idx]})")

In [1]:
import torch
import numpy as np
import random
import torchaudio
import os
import glob
from pathlib import Path

# --- SET YOUR KAGGLE PATHS ---
INPUT_BASE = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
WORKING_BASE = '/kaggle/working'

STEMS_PATH = os.path.join(INPUT_BASE, 'genres_stems')
NOISE_PATH = os.path.join(INPUT_BASE, 'ESC-50-master/audio')
OUTPUT_PATH = os.path.join(WORKING_BASE, 'synthetic_mashups/train')


def seed_everything(seed=42):
    """Locks all random seeds for absolute reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    # If using GPU
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        # Forces deterministic algorithms
        torch.backends.cudnn.deterministic = True 
        torch.backends.cudnn.benchmark = False

# Execute immediately at the top of the script
seed_everything(42)







def generate_synthetic_dataset(stems_dir, noise_dir, output_dir, samples_per_genre=50, target_sr=22050, duration=30):
    """Generates deterministic noisy mashups and saves them to /kaggle/working/."""
    genres = ["blues", "classical", "country", "disco", "hiphop",
"jazz", "metal", "pop", "reggae", "rock"
]
    target_length = target_sr * duration
    
    # Get noise files from read-only input
    noise_files = glob.glob(os.path.join(noise_dir, '**', '*.wav'), recursive=True)
    
    for genre in genres:
        # Create output directories in the writable /kaggle/working/ directory
        genre_out_dir = Path(output_dir) / genre
        genre_out_dir.mkdir(parents=True, exist_ok=True)
        
        song_folders = glob.glob(os.path.join(stems_dir, genre, '*'))
        if not song_folders: 
            print(f"Warning: No songs found for genre {genre}")
            continue
        
        for i in range(samples_per_genre):
            chosen_songs = random.sample(song_folders, 4)
            stems = []
            stem_types = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
            
            for song, stem_type in zip(chosen_songs, stem_types):
                stem_path = os.path.join(song, stem_type)
                if os.path.exists(stem_path):
                    waveform, sr = torchaudio.load(stem_path)
                    
                    # Basic Resampling check (if needed)
                    if sr != target_sr:
                        resampler = torchaudio.transforms.Resample(sr, target_sr)
                        waveform = resampler(waveform)

                    if waveform.shape[1] > target_length:
                        waveform = waveform[:, :target_length]
                    elif waveform.shape[1] < target_length:
                        waveform = torch.nn.functional.pad(waveform, (0, target_length - waveform.shape[1]))
                    stems.append(waveform)
            
            if len(stems) == 4:
                mashup = torch.stack(stems).sum(dim=0)
                mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)
                
                noise_file = random.choice(noise_files)
                noise, _ = torchaudio.load(noise_file)
                
                if noise.shape[1] > target_length:
                    noise = noise[:, :target_length]
                    
                start_idx = random.randint(0, target_length - noise.shape[1])
                intensity = random.uniform(0.1, 0.4)
                
                mashup[:, start_idx:start_idx + noise.shape[1]] += (noise * intensity)
                mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)
                
                # Save to /kaggle/working/
                out_path = genre_out_dir / f"mashup_{i:03d}.wav"
                torchaudio.save(str(out_path), mashup, target_sr)

# Run the generation
generate_synthetic_dataset(STEMS_PATH, NOISE_PATH, OUTPUT_PATH, samples_per_genre=50)

/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be r

In [2]:
from pathlib import Path

# Define the target directory
output_path = Path('/kaggle/working/synthetic_mashups/train')

# Count all .wav files recursively
wav_files = list(output_path.rglob('*.wav'))
total_count = len(wav_files)

# Print results with a quick breakdown per genre
print(f"Total .wav files found: {total_count}")

for genre_dir in sorted(output_path.iterdir()):
    if genre_dir.is_dir():
        count = len(list(genre_dir.glob('*.wav')))
        print(f"  - {genre_dir.name}: {count} files")

Total .wav files found: 500
  - blues: 50 files
  - classical: 50 files
  - country: 50 files
  - disco: 50 files
  - hiphop: 50 files
  - jazz: 50 files
  - metal: 50 files
  - pop: 50 files
  - reggae: 50 files
  - rock: 50 files

✅ Success: All 500 files are present.


In [3]:
import torchaudio
waveform, sr = torchaudio.load('/kaggle/working/synthetic_mashups/train/blues/mashup_000.wav')
print(waveform.shape) 

torch.Size([2, 661500])


In [5]:
import os
import glob
import torch
import torchaudio
from pathlib import Path

def extract_and_save_features(input_dir, output_dir, target_sr=22050):
    """Converts audio to Mel-spectrograms in dB and saves as PyTorch tensors."""
    mel_transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=target_sr, n_fft=2048, hop_length=512, n_mels=128
    )
    amplitude_to_db = torchaudio.transforms.AmplitudeToDB()

    # Find all .wav files in the input directory
    wav_files = glob.glob(os.path.join(input_dir, '**', '*.wav'), recursive=True)
    
    if not wav_files:
        print(f"Warning: No .wav files found in {input_dir}")
        return

    for wav_path in wav_files:
        # Replicate directory structure
        rel_path = os.path.relpath(wav_path, input_dir)
        out_path = Path(output_dir) / rel_path
        out_path = out_path.with_suffix('.pt')
        
        # Ensure the target directory exists in /kaggle/working/
        out_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Process and save
        waveform, sr = torchaudio.load(wav_path)
        mel_spec = mel_transform(waveform)
        mel_spec_db = amplitude_to_db(mel_spec)
        
        torch.save(mel_spec_db, out_path)
    
    print(f"Successfully saved {len(wav_files)} feature files to {output_dir}")


INPUT_DIR = '/kaggle/working/synthetic_mashups/train'
OUTPUT_DIR = '/kaggle/working/features/train'

extract_and_save_features(INPUT_DIR, OUTPUT_DIR)

Successfully saved 500 feature files to /kaggle/working/features/train


In [6]:
import torch

# Load a specific processed file
feature_path = '/kaggle/working/features/train/blues/mashup_000.pt'
features = torch.load(feature_path)

print(f"Feature Shape: {features.shape}")

Feature Shape: torch.Size([2, 128, 1292])


In [7]:
import torch
import torch.nn as nn

class CRNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        
        # TODO 1: CNN Backbone
        # Input shape: (Batch, 1, 128, Time)
        self.cnn = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2), # Shape becomes (32, 64, Time/2)
            
            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)  # Shape becomes (64, 32, Time/4)
        )
        
        # TODO 2: RNN Component
        # 64 channels * 32 frequency bins = 2048
        self.lstm = nn.LSTM(
            input_size=2048, 
            hidden_size=64, 
            num_layers=1, 
            bidirectional=True, 
            batch_first=True
        )
        
        # TODO 3: The Classifier
        # Bidirectional hidden_size=64 -> 64 * 2 = 128 input features
        self.fc = nn.Linear(64 * 2, num_classes)

    def forward(self, x):
        # TODO 4: CNN Forward
        # x shape: (Batch, 1, 128, Time)
        x = self.cnn(x) 
        # After CNN: (Batch, 64, 32, Time_reduced)
        
        # TODO 5: Bridge the gap (Shape Manipulation)
        b, c, f, t = x.size()
        # Permute to (Batch, Time_reduced, Channels, Mels)
        x = x.permute(0, 3, 1, 2).contiguous()
        # Reshape to (Batch, Time_reduced, Channels * Mels)
        x = x.view(b, t, c * f)
        
        # TODO 6: LSTM Forward
        # x shape: (Batch, Time_reduced, 2048)
        lstm_out, _ = self.lstm(x)
        # lstm_out shape: (Batch, Time_reduced, 128)
        
        # TODO 7: Global Max Pooling over time
        # We take the max across the time dimension (dim=1)
        # torch.max returns (values, indices), we only want [0]
        x, _ = torch.max(lstm_out, dim=1)
        # x shape: (Batch, 128)
        
        # TODO 8: Final Prediction
        logits = self.fc(x)
        
        return logits

# --- QUICK TEST ---
# model = CRNN(num_classes=10)
# dummy_input = torch.randn(8, 1, 128, 1292) # Batch of 8
# output = model(dummy_input)
# print(output.shape) # Should be (8, 10)

In [8]:
import torch

# Input matching your established batch and spectrogram size
x = torch.randn(32, 1, 128, 1292)

# Simulate Block 1
x = torch.nn.functional.max_pool2d(x, 2) # (32, 1, 64, 646) - ignoring channels for now

# Simulate Block 2
x = torch.nn.functional.max_pool2d(x, 2) # (32, 1, 32, 323)

print(f"Shape after two MaxPools: {x.shape}")

Shape after two MaxPools: torch.Size([32, 1, 32, 323])


In [9]:
import torch
import torch.nn as nn

# Define the LSTM exactly as in your CRNN
lstm = nn.LSTM(
    input_size=2048, 
    hidden_size=64, 
    num_layers=1, 
    bidirectional=True, 
    batch_first=True
)

# Calculate parameters
params = sum(p.numel() for p in lstm.parameters() if p.requires_grad)

print(f"Total trainable parameters in LSTM: {params}")
# Output: 1073152

Total trainable parameters in LSTM: 1082368


In [13]:
import glob
import os
from pathlib import Path
import torch
from torch.utils.data import Dataset

class PrecomputedFeatureDataset(Dataset):
    def __init__(self, features_dir):
        # Find all .pt files in the directory tree
        self.files = glob.glob(os.path.join(features_dir, '**', '*.pt'), recursive=True)
        self.genres = sorted(['blues', 'classical', 'country', 'disco', 'hiphop', 
                             'jazz', 'metal', 'pop', 'reggae', 'rock'])
        self.genre_to_idx = {g: i for i, g in enumerate(self.genres)}

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_path = self.files[idx]
        # Extract genre from the folder name (e.g., 'pop' from 'features/train/pop/mashup_001.pt')
        genre = Path(file_path).parent.name
        label = self.genre_to_idx[genre]
        
        # Load the precomputed Mel-spectrogram tensor
        feature = torch.load(file_path)
        
        # Standardize shape to (1, 128, 1292)
        # If it's stereo (2 channels), take the mean. If it's missing channel dim, add it.
        if feature.dim() == 3 and feature.shape[0] == 2:
            feature = torch.mean(feature, dim=0, keepdim=True)
        elif feature.dim() == 2:
            feature = feature.unsqueeze(0)
            
        return feature, label

In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

# Now your configuration will work:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CRNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 1. Define your paths (ensure these match where your .pt files are)
TRAIN_FEATURES_DIR = '/kaggle/working/features/train'
# Note: If you haven't made a val set yet, you can use the same for testing 
# or split the train_dataset.
VAL_FEATURES_DIR = '/kaggle/working/features/train' 

# 2. Instantiate the Dataset objects
train_dataset = PrecomputedFeatureDataset(TRAIN_FEATURES_DIR)
val_dataset = PrecomputedFeatureDataset(VAL_FEATURES_DIR)

# 3. Now the Loaders will work
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"Loaded {len(train_dataset)} training samples.")
# Ensure train_dataset and val_dataset are initialized before this:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
num_epochs = 10

# Basic Training Loop
for epoch in range(num_epochs):
    model.train()
    for features, labels in train_loader:
        features, labels = features.to(device), labels.to(device)
        
        # If features are (Batch, 2, 128, 1292), convert to mono
        if features.shape[1] == 2:
            features = torch.mean(features, dim=1, keepdim=True)
            
        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    print(f"Epoch [{epoch+1}/{num_epochs}] completed.")

Loaded 500 training samples.
Epoch [1/10] completed.
Epoch [2/10] completed.
Epoch [3/10] completed.
Epoch [4/10] completed.
Epoch [5/10] completed.
Epoch [6/10] completed.
Epoch [7/10] completed.
Epoch [8/10] completed.
Epoch [9/10] completed.
Epoch [10/10] completed.
